# 20 — Data Skew Pattern

Purpose: handle skewed keys that cause performance issues.

Process:

DATA (skewed)
  |
  v
DETECT SKEW
  |
  v
MITIGATE (salting / repartition)

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("data-skew-pattern")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions","8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

## Step 1 — Skewed data

In [2]:
data = [("u1", i) for i in range(90)] + [(f"u{i}", i) for i in range(2,12)]

df = spark.createDataFrame(data, ["user_id","value"])
df.groupBy("user_id").count().orderBy(F.desc("count")).show()

[Stage 0:>                                                          (0 + 2) / 2]

+-------+-----+
|user_id|count|
+-------+-----+
|     u1|   90|
|    u10|    1|
|     u3|    1|
|     u4|    1|
|     u5|    1|
|     u2|    1|
|     u7|    1|
|     u6|    1|
|     u8|    1|
|     u9|    1|
|    u11|    1|
+-------+-----+



## Step 2 — Problem

In [3]:
# skew join simulation
other = spark.createDataFrame(
    [(f"u{i}", f"attr{i}") for i in range(1,12)],
    ["user_id","attr"]
)

joined = df.join(other, "user_id")
joined.count()

100

## Step 3 — Salting

In [4]:
salted = df.withColumn("salt", (F.rand()*5).cast("int"))

other_salted = other.crossJoin(
    spark.range(0,5).withColumnRenamed("id","salt")
)

joined_salted = salted.join(other_salted, ["user_id","salt"])

joined_salted.count()

100

## Step 4 — Control

In [5]:
print("original rows:", df.count())
print("salted rows:", joined_salted.count())

original rows: 100
salted rows: 100


In [6]:
spark.stop()